# Calculating the rate of kinds of conversations in the massive english dataset.

In [1]:
import os
import numpy as np
import pandas as pd

from scipy.stats import chisquare

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = '../english/4-POWER-LAW-SCALING/reports'

parameter_values = os.path.join(DATA_PATH,'study-level-statistics.csv')
alpha_values = os.path.join(DATA_PATH, 'all.csv')

In [3]:
df = pd.read_csv(alpha_values)
print(df.shape)
df.head()

(23894, 7)


,corpus,cid,speaker,self,a,b,k
0,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-KB0PSUN,False,2.240313e-04,-0.727759,1066
1,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-KB0PSUN,True,1.439414e-08,0.363082,136
2,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS002,False,1.320233e-03,-0.615558,3107
3,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS002,True,1.493614e-03,-0.797152,2556
4,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS006,False,5.985318e-04,-0.503047,2966


In [4]:
dfp = pd.read_csv(parameter_values)
dfp = dfp.rename(columns={'b': 'b_'})
dfp.head()

,self,b_,std,k,se,tstat,p
0,False,-0.387600,0.943652,12019,0.008608,-45.030446,0.0
1,True,-0.486513,1.104611,10719,0.010669,-45.599785,0.0


In [5]:
df = pd.merge(
    left=df,left_on=['self'],
    right=dfp[['self', 'b_']], right_on=['self'],
    how='left'
)

In [6]:
df['faster'] = df['b'] < df['b_']

## Hypothesis test: Conversation Types are distributed equally.

We wanted to test the assumption that all possible combinations of information scaling factors are equal. We thus tested

$(\alpha_s < \bar{\alpha_s})\ \ \&\ \ (\alpha_o < \bar{\alpha_o})$ (Both are faster)

$(\alpha_s < \bar{\alpha_s})\ \ \&\ \ (\alpha_o > \bar{\alpha_o})$ (Self is fast, other is slow)

$(\alpha_s > \bar{\alpha_s})\ \ \&\ \ (\alpha_o < \bar{\alpha_o})$ (Self is slow, other is fast)

$(\alpha_s > \bar{\alpha_s})\ \ \&\ \ (\alpha_o > \bar{\alpha_o})$ (Both are slower)

In [7]:
df['self'].sum()

np.int64(11290)

In [8]:
dfo = pd.merge(
    left=df.loc[df['self']], left_on=['corpus', 'cid', 'speaker'],
    right=df[['corpus', 'cid', 'speaker', 'faster']].loc[~df['self']], right_on=['corpus', 'cid', 'speaker'],
    how='left'
)
print(dfo.shape)
dfo.isna().sum()

(11290, 10)


corpus       0
cid          0
speaker      0
self         0
a            0
b            0
k            0
b_           0
faster_x     0
faster_y    49
dtype: int64

In [16]:
dfo['corpus'].unique()

array(['CABNC', 'callfriend-eng_n', 'callfriend-eng_s', 'callhome',
       'CANDOR', 'GCSAusE', 'SBCSAE', 'SCoSE', 'grace', 'DISPEL',
       'Frederiksen', 'Graesser', 'med_school', 'MICASE', 'CORAAL'],
      dtype=object)

In [9]:
dfo = dfo.loc[~dfo['faster_y'].isna()]
dfo.head()

,corpus,cid,speaker,self,a,b,k,b_,faster_x,faster_y
0,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-KB0PSUN,True,1.439414e-08,0.363082,136,-0.486513,False,True
1,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS002,True,1.493614e-03,-0.797152,2556,-0.486513,True,True
2,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS006,True,3.587024e-04,-0.430773,1953,-0.486513,False,True
3,CABNC,CABNC-KB0RE001-KB0,CABNC-CABNC-KB0RE001-KB0-KB0PSUN,True,5.573154e-06,-0.930984,105,-0.486513,True,True
4,CABNC,CABNC-KB0RE001-KB0,CABNC-CABNC-KB0RE001-KB0-PS005,True,5.494917e-04,0.090868,435,-0.486513,False,False


In [10]:
obs = np.array([
    ((dfo['faster_x']==True) & (dfo['faster_y']==True)).sum(),
    ((dfo['faster_x']==True) & (dfo['faster_y']==False)).sum(),
    ((dfo['faster_x']==False) & (dfo['faster_y']==True)).sum(),
    ((dfo['faster_x']==False) & (dfo['faster_y']==False)).sum(),
])
obs

array([3483, 1434, 2385, 3939])

In [11]:
obs/obs.sum()

array([0.30984788, 0.12756872, 0.21216974, 0.35041366])

In [12]:
exp = np.array([obs.mean()]*len(obs))
exp, exp.sum()

(array([2810.25, 2810.25, 2810.25, 2810.25]), np.float64(11241.0))

In [13]:
chisquare(f_obs=obs, f_exp=exp)

Power_divergenceResult(statistic=np.float64(1352.751801441153), pvalue=np.float64(5.266819521910178e-293))

## Hypothesis test: CANDOR is equivalent to the rest of the study

In [18]:
candor_obs = np.array([
    ((dfo['faster_x'].loc[dfo['corpus'].isin(['CANDOR'])]==True) & (dfo['faster_y'].loc[dfo['corpus'].isin(['CANDOR'])]==True)).sum(),
    ((dfo['faster_x'].loc[dfo['corpus'].isin(['CANDOR'])]==True) & (dfo['faster_y'].loc[dfo['corpus'].isin(['CANDOR'])]==False)).sum(),
    ((dfo['faster_x'].loc[dfo['corpus'].isin(['CANDOR'])]==False) & (dfo['faster_y'].loc[dfo['corpus'].isin(['CANDOR'])]==True)).sum(),
    ((dfo['faster_x'].loc[dfo['corpus'].isin(['CANDOR'])]==False) & (dfo['faster_y'].loc[dfo['corpus'].isin(['CANDOR'])]==False)).sum(),
])
candor_obs

array([1651,   77,  995,  589])

In [19]:
exp = (obs/obs.sum()) * candor_obs.sum()
exp

array([1026.21617294,  422.50760608,  702.70616493, 1160.57005604])

In [20]:
chisquare(f_obs=candor_obs,f_exp=exp)

Power_divergenceResult(statistic=np.float64(1065.9970814871297), pvalue=np.float64(8.667618682365903e-231))

## Other hypothesis tests

In [ ]:
obs = df[['self', 'faster']].value_counts(sort=False)
obs

In [ ]:
key_names = set([k[0] for k in obs.keys()])
key_names

#### Hyp: all possible combos are equal

In [ ]:
exp = np.array([obs.mean()]*len(obs.values))
exp

In [ ]:
chisquare(f_obs=obs.values, f_exp=exp)

#### Hyp: faster and slower are the same within each group

In [ ]:
# hyp: within self and other rates
exp = [[obs[k].sum()/len(obs[k])] * len(obs[k]) for k in key_names]
exp = np.array(sum(exp, []))
exp

In [ ]:
chisquare(f_obs=obs.values, f_exp=exp)